# Lab 3 — Ray Data Fundamentals (Fundamentals)

**Time:** ~15–20 min  
**Mode:** Complete by hand.

## Learning objectives
1. Read Parquet data with `ray.data.read_parquet` and inspect with `take`/`take_batch`.
2. Apply `map`, `map_batches`, and `filter` (both lambda and `col` expressions).
3. Run a small aggregation and write the output to cluster storage.

## Dataset
`s3://anyscale-public-materials-use2/ecom/catalog` — eCom product catalog. Each record has fields like
`manuf_prod_id`, `prod_id`, `cat`, `name`, `desc`, `price` (the precise schema
is visible after the read in Exercise 1).

## Setup

In [ ]:
import ray
from ray.data.expressions import col

if not ray.is_initialized():
    ray.init()

CATALOG = 's3://anyscale-public-materials-use2/ecom/catalog'
SCRATCH = '/mnt/cluster_storage/lab_2_1_output'

## Exercise 1 — Read & inspect
1. Read the catalog Parquet into a Dataset.
2. Print the row count.
3. Print the schema.
4. Print a single sample record using `take(1)`.


In [ ]:
# TODO: read & inspect the catalog


## Exercise 2 — Clean and shorten field names
Some of the source field names are verbose. Use a `map` with a function (or a
dict-rename helper of your choosing) so that the output rows have these short
field names: `mfg_id`, `item_id`, `cat`, `name`, `desc`, `price`. Drop any
extra fields. Verify with `take(1)`.


In [ ]:
# TODO: rename / project columns


## Exercise 3 — Filter two ways
Build two filtered datasets:
1. Using a Python lambda — keep only rows where `cat == 'Housewares'`.
2. Using a `col` expression — keep only rows where `price > 50`.

Print the count of each.

*Note:* The `col`-expression form lets the planner push the filter down
into the Parquet reader, which can be much faster on large datasets.

In [ ]:
# TODO: filter two ways


## Exercise 4 — Aggregate and write
Compute the **average price per category** using `groupby` and `mean`.
Write the cleaned, renamed dataset (from Exercise 2) to Parquet at `SCRATCH`.
Confirm the write by reading it back and printing the row count.


In [ ]:
# TODO: aggregate and write


---

# Solutions

### Solution 1

In [ ]:
ds = ray.data.read_parquet(CATALOG)
print('count:', ds.count())
print('schema:', ds.schema())
ds.take(1)

### Solution 2

In [ ]:
renames = {
    'manufacturer_product_number': 'mfg_id',
    'ecom_vendor_product_number': 'item_id',
    'category' : 'cat',
    'item_name' : 'name',
    'item_description' : 'desc',
    'price_usd' : 'price'
}
# mfg_id, item_id, cat, name, desc, price. 
def rename_row(row):
    out = {}
    for k, v in row.items():
        out[renames.get(k, k)] = v
    return {k: out[k] for k in ('mfg_id', 'item_id', 'cat', 'name', 'desc', 'price') if k in out}

cleaned = ray.data.read_parquet(CATALOG).map(rename_row)
cleaned.take(1)

### Solution 3

In [ ]:
lambda_filtered = cleaned.filter(lambda r: r['cat'] == 'Housewares')
print('Housewares (lambda):', lambda_filtered.count())

expr_filtered = cleaned.filter(expr=(col('price') > 50))
print('price > 50 (col expr):', expr_filtered.count())

### Solution 4

In [ ]:
avg_by_cat = cleaned.groupby('cat').mean('price')
avg_by_cat.show()

cleaned.write_parquet(SCRATCH, mode=ray.data.SaveMode.OVERWRITE)
print('rows written:', ray.data.read_parquet(SCRATCH).count())

## Wrap-up
Things to notice:
- `take` / `take_batch` are your friends for inspecting intermediate state.
- Lambda filters are flexible; `col`-expression filters are faster on Parquet
  because they can be pushed down into the reader.
- `cluster_storage` is shared across nodes within the cluster but is **ephemeral**
  to the cluster lifetime — use `user_storage` or shared S3 for outputs you need
  to keep beyond or outside the workspace lifetime.